# 911 Calls Wav2Vec2
https://www.kaggle.com/code/ajax0564/video-caption-using-wavetovec-2-0<br/>
https://www.kaggle.com/code/stpeteishii/english-audio-wav2vec2

A "911 call" refers to a telephone call made to the emergency telephone number 911 in the United States and several other countries. The number 911 is a universal emergency number that allows people to quickly reach emergency services, such as police, fire departments, and medical services. When someone dials 911, their call is routed to a local emergency dispatch center, also known as a Public Safety Answering Point (PSAP), where trained operators or dispatchers assess the situation and send appropriate help to the caller's location.

## facebook/wav2vec2-base-960h
The facebook/wav2vec2-base-960h is a speech recognition model from Facebook AI, trained on a massive dataset of unlabeled speech audio. It is a Transformer-based model with 960 hidden units, and it can be used to transcribe speech into text.

The model was pre-trained on 53,000 hours of unlabeled speech audio from the Librispeech dataset. It was then fine-tuned on 10 minutes of labeled speech audio from the same dataset. This results in a model that can achieve a word error rate (WER) of 4.8% on the Librispeech test set.

The facebook/wav2vec2-base-960h model can be used for a variety of speech recognition tasks, such as:

Transcribing audio recordings of meetings or lectures
Translating audio recordings of foreign language speech
Creating closed captioning for videos
Developing virtual assistants that can understand spoken commands
The model is available on the Hugging Face Hub, and it can be used with a variety of frameworks, such as PyTorch and TensorFlow.

In [3]:
# Dependencies are already installed in this project's .venv.
# If running on Kaggle, uncomment the next line:
# !pip install -q openai-whisper librosa pandas

In [4]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from analyze_audio import run_pipeline

ModuleNotFoundError: No module named 'transformers'

In [ ]:
data=pd.read_csv('911_first6sec/911_metadata.csv')
print(f'Metadata rows: {len(data)}')
print(f'Unique audio files: {data["filename"].nunique()}')
display(data[['id', 'title', 'state', 'filename']].head())

In [ ]:
missing_files = [name for name in data['filename'] if not Path(name).is_file()]
assert not missing_files, f'Missing {len(missing_files)} audio files'
print('All metadata audio paths are valid.')

In [ ]:
results = run_pipeline(
    dataset_dir=Path('911_first6sec'),
    output_csv=Path('911_audio_analysis.csv'),
    detailed_csv=Path('911_audio_analysis_detailed.csv'),
    checkpoint=Path('transcription_checkpoint.json'),
    model_name='tiny.en',
)
display(results.head(10))

In [ ]:
print(f'Output rows: {len(results)}')
print('\nSentiment distribution:')
display(results['Sentiment'].value_counts().to_frame('Calls'))
print('\nExtracted event distribution:')
display(results['Extracted_Event'].value_counts().to_frame('Calls'))

In [ ]:
high_urgency = results.sort_values('Urgency_Score', ascending=False).head(20)
display(high_urgency)

In [ ]:
assert list(results.columns) == [
    'Call_ID', 'Transcript', 'Extracted_Event',
    'Location', 'Sentiment', 'Urgency_Score'
]
assert results['Urgency_Score'].between(0, 1).all()
assert results['Call_ID'].is_unique
print('Output validation passed.')

In [ ]:
print('Required submission file: 911_audio_analysis.csv')
print('Audit file with names and urgency phrases: 911_audio_analysis_detailed.csv')